## Lab 3 Part 2 - EECE.4860/5860 at UMass Lowell

Complete the missing code to create a new tool based on RAG, and execute a complex query that invokes the tools.

This example uses LangChain APIs.

In [ ]:
%pip install --upgrade langchain langgraph langchain-community langchain-huggingface langchain_openai wikipedia numexpr pypdf 

In [ ]:
from langchain_core.tools import tool

@tool
def magic_function(input: int) -> int:
    """Applies a magic function to an input."""
    return input + 2

tools = [magic_function]

query = "what is the value of magic_function(3)?"

In [ ]:
from typing import Annotated
from langchain_core.tools import tool
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.indexes import VectorstoreIndexCreator
from langchain_huggingface import HuggingFaceEmbeddings

top_k = 1
chunk_size=2000
overlap=200

# The new tool you need to create is a RAG tool that answer queries about DeepSeek-R1
# The tool leverages a few APIs that you see in the simple_rag.ipynb example, including
# RecursiveCharacterTextSplitter(), VectorstoreIndexCreator(), vectorstore.similarity_search()
#
loader = PyPDFLoader("https://arxiv.org/pdf/2501.12948")
docs = loader.load()
# TODO: call  RecursiveCharacterTextSplitter() using chunk_size and overlap
# text_splitter = ...

# TODO: create the vector database by calling VectorstoreIndexCreator()
# TODO: using HuggingFaceEmbedding() and "intfloat/multilingual-e5-large-instruct" model
#index = VectorstoreIndexCreator(
#    embedding= ...,
#    text_splitter=...
#).from_loaders([loader])


@tool
def query_deepseek_r1_paper(input: Annotated[str, "Query relating to DeepSeek-R1"]) -> str:
    """Provides context that can be used to answer questions on DeepSeek-R1"""
    
    print(f"Received input query: {input}\n")

    # TODO: perform similarity search on the input and top_k
    # results = ...
    
    if not results:
        print("No results found for the query.\n\n")
        return "no results"
    
    # TODO: concatenate the results using "\n" as a separator
    # context = 

    if context:
        print(f"Context found:\n{context}\n\n")
        
    return context
 

In [ ]:
# We will use an OpenAI model hosted by Jetstream2 for our agent (an API key is not needed when running on a Jetstream2 instance)
from langchain_openai import ChatOpenAI

model = ChatOpenAI(base_url="http://llm.jetstream-cloud.org/gpt-oss-120b/v1/",
                   model="gpt-oss-120b",
                   api_key="empty",
                   temperature=0.1,
                   max_tokens=1000,
                   timeout=30)


In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
import numexpr as ne


api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=1000
)
wikipedia_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

@tool
def calculator(expression: str) -> str:
    """A tool for solving mathematical problems. Input must be a mathematical expression in Python syntax."""
    print(expression)
    print(ne.evaluate(expression).item())
    return str(ne.evaluate(expression).item())

tools=[wikipedia_tool, calculator, magic_function, query_deepseek_r1_paper]

In [ ]:
from langchain.agents import create_agent

agent = create_agent(llm_nvidia, tools)

# Increase the recursion limit in the configuration
config = {
   "recursion_limit": 20  # Increase the limit as needed
}

query1 = "In what year was the film Departed with Leopnardo Dicaprio released? What is this year raised to the power of 0.43?"

query2 = "Use the year the film Departed with Leopnardo Dicaprio was released to calculate the value of magic_function."

query3 = "What are the main limitations of DeepSeek-R1?"

messages1 = agent.invoke({"messages": [("human", query1)]}, config=config)

print('-'*200)
print("Query 1:", query1)
print('-'*200)
print("Response 1:", messages1["messages"][-1].content)
print('-'*200, '\n')

messages2 = agent.invoke({"messages": [("human", query2)]}, config=config)
print('-'*200)
print("Query 2:", query2)
print('-'*200)
print("Response 2:", messages2["messages"][-1].content)
print('-'*200, '\n')


messages3 = agent.invoke({"messages": [("human", query3)]}, config=config)
print('-'*200)
print("Query 3:", query3)
print('-'*200)
print("Response 3:", messages3["messages"][-1].content)
print('-'*200, '\n')